Report 5


Anh Do

020416-2317

anhd@kth.se

In [180]:
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

## Code

### ECP

In [181]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [182]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.set_options(WLS)
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.6f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -12.000000   | 27.000000   
2     | -10.000000   | 27.000000   
3     | -9.583333    | 24.395833   
4     | -8.750000    | 19.062500   
5     | -8.750000    | 11.062500   
6     | -7.760417    | 12.557617   
7     | -7.414279    | 17.607152   
8     | -7.340979    | 7.384568    
9     | -6.871026    | 17.870325   
10    | -6.743382    | 7.335322    
11    | -6.665778    | 5.574822    
12    | -6.448804    | 6.055243    
13    | -6.427411    | 4.819187    
14    | -6.244850    | 3.341987    
15    | -6.035800    | 7.548069    
16    | -5.927437    | 3.082011    
17    | -5.579046    | 3.343273    
18    | -5.573984    | 4.257775    
19    | -5.536929    | 2.241301    
20    | -5.453806    | 3.354415    
21    | -5.390747    | 3.257580    
22    | -5.356249    | 1.833128    
23    | -5.288019    | 2.181599    
24    | -5.223485    | 5.386811    
25    | -5.197924    | 1.738826    
26    | -5.174643    | 

### OA

In [183]:
def create_nlp_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])

    # Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= 0)

    return m

def create_feas_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))

    m.u = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.u, sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])
        
    # Relaxed Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= m.u)
    return m

def create_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

        
    m.mu = pyo.Var(domain=pyo.Reals)
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    #m.obj = pyo.Objective(expr=-sum([m.x[i] for i in m.I]), sense=pyo.minimize)

    # My lazy cut so i dont need to handle vectorized constraints that is always the strongest (static cut) because we have linearity
    m.static_cut = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    
    m.feas_cuts = pyo.ConstraintList()
    m.ubd_cuts = pyo.ConstraintList()
    m.infeas_cuts = pyo.ConstraintList()
    return m

In [184]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as master_solver:
    master_solver.set_options(WLS)
    master_solver.options.update(grb_params)
    master_model = create_master_model()
    
    tol = 1e-4
    max_iter = 100
    iteration = 0
    guess = {5: 1, 6: 1, 7: 1, 8: 0} # Initial Integer Guess
    prev_guess = None

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Status':<12} | {'y_vals':<20}")
    print("-" * 60)

    while iteration < max_iter:
        master_solver.set_instance(master_model)
        iteration += 1
        
        prev_guess = guess.copy()
        
        # --- 1. Solve NLP Subproblem ---
        NLP_model = create_nlp_model(guess)
        NLP_solver = SolverFactory('gurobi_direct')
        result_nlp = NLP_solver.solve(NLP_model, load_solutions=False)

        if result_nlp.solver.termination_condition == pyo.TerminationCondition.optimal:
            NLP_model.solutions.load_from(result_nlp)
            f_k = pyo.value(NLP_model.obj)
            x_k = {i: pyo.value(NLP_model.x[i]) for i in NLP_model.I}
            g_k = sum(x_k[i]**2 for i in NLP_model.I) - 3

            # Update Primal Bound
            current_ubd = master_model.mu.ub if master_model.mu.has_ub() else float('inf')
            print(f_k)
            if f_k < current_ubd:
                master_model.mu.setub(f_k)
                x_opt = x_k.copy()
            
            print(f"{iteration:<5} | {f_k:<12.4f} | {'Feasible':<12} | {str(guess):<20}")

            # Add Optimality Cut: mu >= f(x^k) + ∇f(x^k)^T (x - x^k)
            # ∇f = -1 for all i, so: mu >= f_k - sum(x_i - x_k_i)
            master_model.ubd_cuts.add(
                expr= master_model.mu >= f_k + sum([(-1)*(master_model.x[i] - x_k[i]) for i in master_model.I])
            )
            master_solver.add_constraint(master_model.ubd_cuts[len(master_model.ubd_cuts)])

            # Add Feasibility Cut: g(x^k) + ∇g(x^k)^T (x - x^k) <= 0
            # ∇g = 2*x^k for each i
            master_model.feas_cuts.add(
                expr= 0 >= g_k + sum([(2*x_k[i])*(master_model.x[i] - x_k[i]) for i in master_model.I])
            )
            master_solver.add_constraint(master_model.feas_cuts[len(master_model.feas_cuts)])

        elif result_nlp.solver.termination_condition == pyo.TerminationCondition.infeasible:
            print(f"{iteration:<5} | {'--':<12} | {'Infeasible':<12} | {str(guess):<20}")
           
            # --- 2. Solve Feasibility Problem (Relaxation) ---
            FEAS_model = create_feas_model(guess)
            FEAS_solver = SolverFactory('gurobi_direct')
            result_feas = FEAS_solver.solve(FEAS_model, load_solutions=False)
            FEAS_model.solutions.load_from(result_feas)

            x_k = {i: pyo.value(FEAS_model.x[i]) for i in FEAS_model.I}
            g_k = sum((x_k[i]**2 for i in FEAS_model.I)) - 3

            # Add Infeasibility Cut: g(x^k) + ∇g(x^k)^T (x - x^k) <= 0
            master_model.infeas_cuts.add(
                expr= 0 >= g_k + sum([(2*x_k[i])*(master_model.x[i] - x_k[i]) for i in master_model.I])
            )
            master_solver.add_constraint(master_model.infeas_cuts[len(master_model.infeas_cuts)])

        else:
            raise RuntimeError(f"Unexpected NLP status: {result_nlp.solver.termination_condition}")

        # --- 3. Solve Master Problem ---
        result_master = master_solver.solve()
        if result_master.solver.termination_condition == pyo.TerminationCondition.infeasible:
            print("Master Infeasible -> Last solution was optimal.")
            break
        
        # --- 4. Update Integer Guess ---
        # Rounding is critical for integer vars returned by solvers
        guess = {i: int(round(pyo.value(master_model.x[i]))) for i in range(5, 9)}
    
    print("-" * 60)
    print("Final Solution:")
    print(f"Objective: {pyo.value(master_model.mu.ub):.6f}")
    for i in master_model.I:
        print(f"x[{i}] = {x_opt[i]}")

Iter  | Obj Value    | Status       | y_vals              
------------------------------------------------------------
-3.0
1     | -3.0000      | Feasible     | {5: 1, 6: 1, 7: 1, 8: 0}
2     | --           | Infeasible   | {5: 0, 6: 3, 7: 0, 8: 3}
3     | --           | Infeasible   | {5: 0, 6: 0, 7: 0, 8: 3}
4     | --           | Infeasible   | {5: 0, 6: 3, 7: 0, 8: 0}
5     | --           | Infeasible   | {5: 0, 6: 1, 7: 0, 8: 2}
6     | --           | Infeasible   | {5: 0, 6: 2, 7: 0, 8: 1}
7     | --           | Infeasible   | {5: 0, 6: 0, 7: 0, 8: 2}
8     | --           | Infeasible   | {5: 0, 6: 2, 7: 0, 8: 0}
-4.0
9     | -4.0000      | Feasible     | {5: 0, 6: 1, 7: 0, 8: 1}
10    | --           | Infeasible   | {5: 2, 6: 1, 7: 0, 8: 0}
11    | --           | Infeasible   | {5: 0, 6: 0, 7: 3, 8: 1}
-3.0
12    | -3.0000      | Feasible     | {5: 1, 6: 0, 7: 1, 8: 1}
-3.0
13    | -3.0000      | Feasible     | {5: 1, 6: 0, 7: 1, 8: 1}
-3.0
14    | -3.0000      | Feasible     